# A10 experiment notebook (Kaggle T4 x2): depth vs width, isolated

**Superseded, never actually run.** A10 ended up trained on a rented Vast.ai GPU instead
(`vastai/run_a10.sh` + `vastai/vastai_eval_a10.ipynb`) once that turned out much faster/cheaper
than expected (~9.6x measured vs this Kaggle T4x2 path) -- see `docs/RESEARCH_LOG.md` and
`docs/PROJECT_PLAN.md` for the real results. Kept here as a Kaggle-only fallback if renting
isn't an option; the plan/config below is still accurate, just not what actually produced the
numbers in README.md.

Tests whether *more depth at a fixed width* helps, isolated from the width change that also
happened between `d4` and `d6`. Architecture: `--aspect-ratio=48 --depth=7` -> `model_dim=384`
(**same width as `d6`**), 7 layers instead of `d6`'s 6 -> **87.88M params** (vs `d6`'s 73.53M).
Same `--target-param-data-ratio=20` (Chinchilla-compute-optimal) as every other run so far, for a
clean comparison -- see `docs/RESEARCH_LOG.md` for the full reasoning and the param-count math.

Model tag: `a10` (not `d7`, which would be ambiguous with a *default*-aspect-ratio depth=7 model,
a different architecture with `model_dim=512`).

Reuses the existing tokenizer and cached dataset shards from Drive (same `vocab_size=32768` as
`d4`/`d6` -- only the model shape changes here, not the tokenizer) -- no retraining needed.

**Estimated time: ~5.5h pretrain + SFT + a quick eval** (calibrated from `d4`/`d6`'s actual
measured FLOPs-to-wall-clock rate, not guessed) -- should fit in one 12h Kaggle session with
margin. All three phases run in this one notebook so it can be launched and left unattended;
checkpoints sync to Drive continuously, so a killed session can resume cleanly.

Runs the full `chat_eval.py`/BLiMP suite separately afterward, via `kaggle/kaggle_eval.ipynb`
with `MODEL_TAG="a10"` -- not duplicated here, to keep this notebook's unattended runtime
predictable.

Upload via File -> Upload Notebook. Same 4 Kaggle Secrets, T4 x2 accelerator, internet access.

## Cell 1: clone repo, install dependencies

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = "/kaggle/working/repo"
MODEL_TAG = "a10"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print("Cell 1 done.")

## Cell 2: configure rclone, pull the shared tokenizer + cached dataset + any existing `a10` checkpoint

In [ ]:
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
client_id = secrets.get_secret("GDRIVE_CLIENT_ID").strip()
client_secret = secrets.get_secret("GDRIVE_CLIENT_SECRET").strip()
oauth_token = secrets.get_secret("GDRIVE_OAUTH_TOKEN").strip()
folder_id = secrets.get_secret("GDRIVE_FOLDER_ID").strip()

rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"client_id = {client_id}\n"
        f"client_secret = {client_secret}\n"
        f"token = {oauth_token}\n"
        f"root_folder_id = {folder_id}\n"
        "team_drive =\n"
    )

!rclone lsd gdrive:

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = "/kaggle/working/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

# Tokenizer and dataset are shared with d4/d6 (same vocab_size=32768) -- reuse, don't retrain/redownload.
!rclone copy gdrive:tokenizer {NANOCHAT_BASE_DIR}/tokenizer --checksum -v
!rclone copy gdrive:base_data_climbmix {NANOCHAT_BASE_DIR}/base_data_climbmix --checksum -v

# Resume detection for this specific experiment's checkpoint, if a previous session got partway through.
sys.path.insert(0, REPO_DIR)
from nanochat.checkpoint_manager import find_last_step

base_ckpt_remote = f"{DRIVE_REMOTE}base_checkpoints/{MODEL_TAG}"
base_ckpt_local = os.path.join(NANOCHAT_BASE_DIR, "base_checkpoints", MODEL_TAG)
listing = subprocess.run(["rclone", "lsf", base_ckpt_remote], capture_output=True, text=True)
RESUME_STEP = -1
if listing.returncode == 0 and listing.stdout.strip():
    print(f"Found existing {MODEL_TAG} checkpoint on Drive, downloading...")
    !rclone copy {base_ckpt_remote} {base_ckpt_local} --checksum -v
    try:
        RESUME_STEP = find_last_step(base_ckpt_local)
        print(f"Will resume pretraining from step {RESUME_STEP}.")
    except FileNotFoundError:
        print("Checkpoint dir exists but has no checkpoints in it yet.")
else:
    print(f"No prior {MODEL_TAG} checkpoint found, starting fresh.")
os.environ["RESUME_STEP"] = str(RESUME_STEP)

print("Cell 2 done.")

## Cell 3: pretrain (`--aspect-ratio=48 --depth=7`, background Drive sync)

`--device-batch-size=8` reused from `d6` without a fresh VRAM probe: this architecture has the
*same* `model_dim=384`/`n_head=3` as `d6` (only 1 extra layer), and `d6` had comfortable headroom
at batch 8 (7.96GiB/15GiB used, verified safe up to 13) -- one more layer won't meaningfully
change that. If this turns out wrong and it OOMs, drop to `--device-batch-size=4` (still a clean
divisor of 64, see `kaggle_train.ipynb`'s Cell 1 comment for why that matters) and rerun this cell
-- it'll resume from the last save via `RESUME_STEP`.

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
sync_proc = subprocess.Popen(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "120", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print(f"Started background checkpoint sync watcher, pid={sync_proc.pid}")

resume_step = int(os.environ.get("RESUME_STEP", "-1"))
resume_args = f"--resume-from-step={resume_step}" if resume_step >= 0 else ""

train_cmd = (
    "torchrun --standalone --nproc_per_node=2 -m scripts.base_train -- "
    "--depth=7 --aspect-ratio=48 --window-pattern=L --device-batch-size=8 "
    f"--target-param-data-ratio=20 --save-every=100 {resume_args} --run=dummy --model-tag=a10"
)
print(f"Running: {train_cmd}")
try:
    !{train_cmd}
finally:
    sync_proc.terminate()
    sync_proc.wait()
    !python kaggle/sync_checkpoints.py --remote gdrive: --once --log-file {SYNC_LOG}
    print("Pretrain cell finished (or was interrupted), sync watcher stopped.")

## Cell 4: SFT

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)
MODEL_TAG = "a10"

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
sync_proc = subprocess.Popen(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "120", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print(f"Started background checkpoint sync watcher, pid={sync_proc.pid}")

# Same mixture/cap as the d4/d6 SFT runs: SmolTalk only (no MMLU/GSM8K -- pointless at this
# scale), capped at 500 steps as a safety margin since chat_sft.py only saves at the very end.
sft_cmd = (
    "torchrun --standalone --nproc_per_node=2 -m scripts.chat_sft -- "
    f"--model-tag={MODEL_TAG} --mmlu-epochs=0 --gsm8k-epochs=0 "
    "--num-iterations=500 --chatcore-every=-1 --eval-every=100 --run=dummy"
)
print(f"Running: {sft_cmd}")
try:
    !{sft_cmd}
finally:
    sync_proc.terminate()
    sync_proc.wait()
    !python kaggle/sync_checkpoints.py --remote gdrive: --once --log-file {SYNC_LOG}
    print("SFT cell finished (or was interrupted), sync watcher stopped.")

## Cell 5: quick chat test + repetition metric

Fast sanity check right here -- full `chat_eval.py`/BLiMP comparison against `d4`/`d6` happens
separately via `kaggle/kaggle_eval.ipynb` (set `MODEL_TAG="a10"` there).

In [ ]:
import os

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)
MODEL_TAG = "a10"

!python -m scripts.chat_cli -i sft -g {MODEL_TAG} -p "hi"
!python -m scripts.chat_cli -i sft -g {MODEL_TAG} -p "What is your name?"
!python -m scripts.eval_repetition -i sft -g {MODEL_TAG} --repetition-penalty 1.2 --no-repeat-ngram-size 3